# Banco de pruebas

Aqui se prueba una consulta antes de llevarla al ETL. El notebook no forma
parte del proceso automatico: es el sitio donde se mira el resultado, se
comprueba el grano y se afina la limpieza.

Ejecutad las celdas en orden.

In [ ]:
import sys
from pathlib import Path

# El notebook vive en notebooks/, asi que hay que anadir la raiz al path
# para poder importar src/.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from src.olist_ETL import crear_engine, extraer, comprobar_grano, CONSULTAS
from src.limpieza import limpiar, LIMPIEZA

engine = crear_engine()
print("Conexion lista. Consultas registradas:", list(CONSULTAS))

## 1. Ejecutar una consulta existente

Cambiad `NOMBRE` por cualquiera de las registradas.

In [ ]:
NOMBRE = "catalogo_productos"

df = extraer(engine, NOMBRE)
print(df.shape)
df.head()

## 2. Comprobar el grano

Esta es la comprobacion que el ETL hace en cada ejecucion. Si el numero de
filas no coincide con el de valores distintos de la clave, algo esta
multiplicando filas y el CSV saldria inflado.

In [ ]:
clave = CONSULTAS[NOMBRE]

print(f"filas: {len(df)}")
print(f"{clave} distintos: {df[clave].nunique()}")

comprobar_grano(df, clave, NOMBRE)
print("Grano correcto.")

## 3. Medir antes de limpiar

Una regla de limpieza se justifica con un numero, no con un tutorial.
Antes de escribir nada, contad el defecto.

In [ ]:
# Cuanto hay de cada cosa que os preocupa
print("nulos por columna:")
print(df.isna().sum()[lambda s: s > 0].to_string() or "  ninguno")

print()
print("filas duplicadas enteras:", df.duplicated().sum())
print("filas:", len(df))

## 4. Traer la limpieza del Proyecto III

Vuestro notebook del Proyecto III no se tira: se convierte en funciones.
El cambio es mecanico, un `def` arriba y un `return` abajo:

```python
# Proyecto III, celda suelta
df = df[df["precio"] > 0]
df["ciudad"] = df["ciudad"].str.strip()

# Proyecto IV, funcion en src/limpieza.py
def quitar_precios_a_cero(df):
    df = df[df["precio"] > 0]
    df["ciudad"] = df["ciudad"].str.strip()
    return df
```

Probadla aqui primero. Cuando funcione, la pegais en `src/limpieza.py` y la
registrais en `LIMPIEZA` bajo su dataset. No hay que copiar codigo dos veces:
el notebook importa las mismas funciones que ejecuta el ETL.

In [ ]:
# Escribid aqui vuestra regla y probadla sobre df
def mi_regla(d):
    # ... vuestro codigo del Proyecto III ...
    return d

antes = len(df)
prueba = mi_regla(df.copy())
print(f"{antes} -> {len(prueba)} filas")
prueba.head()

In [ ]:
# Lo que el ETL aplica hoy a este dataset
print([f.__name__ for f in LIMPIEZA[NOMBRE]])

limpio = limpiar(df.copy(), NOMBRE)

print()
print(len(df), "->", len(limpio), "filas")
print("columnas nuevas:", [c for c in limpio.columns if c not in df.columns])
limpio.head()

## 5. Probar SQL suelto

Para iterar sin tocar todavia ningun fichero `.sql`.

In [ ]:
consulta = """
SELECT order_status, COUNT(*) AS pedidos
FROM orders
GROUP BY order_status
ORDER BY pedidos DESC
"""

pd.read_sql(consulta, engine)

## 6. Dejarlo listo para el ETL

**Consulta nueva**
1. Guardadla en `queries/<nombre>.sql`, con el grano declarado en la primera linea.
2. Anadid `"<nombre>": "<columna_de_grano>"` a `CONSULTAS` en `src/olist_ETL.py`.

**Regla de limpieza nueva**
1. Pegad la funcion en `src/limpieza.py`.
2. Anadidla a la lista de su dataset en `LIMPIEZA`.

Y despues, desde la raiz:

```bash
python main.py
```

Donde va cada limpieza: si es un filtro o un valor por defecto, en SQL. Si
necesita mirar el conjunto entero o manipular texto, en `limpieza.py`.